In [1]:
import os

In [3]:
import sagemaker
import boto3
from sagemaker import get_execution_role

sagemaker_session = sagemaker.Session()

region = boto3.Session().region_name
print(region)

us-east-1


In [4]:
from sagemaker import get_execution_role
# Define IAM role
role = get_execution_role()
print(role)

arn:aws:iam::664752154920:role/LabRole


In [5]:
bucket = sagemaker_session.default_bucket()
print(bucket)

sagemaker-us-east-1-664752154920


In [6]:
import pandas as pd
import numpy as np

s3 = boto3.client("s3")

#s3://sagemaker-example-files-prod-us-east-2/datasets/tabular/uci_bank_marketing/bank-additional-full.csv

In [7]:
key = "datasets/tabular/uci_bank_marketing/bank-additional-full.csv"

s3.download_file(
    f"sagemaker-example-files-prod-{region}",
    key,
    "bank-additional-full.csv"
)

In [8]:
df = pd.read_csv("bank-additional-full.csv")

In [9]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [10]:
select = ['age','campaign', 'euribor3m', 'cons.conf.idx','cons.price.idx','y']

In [11]:
df_select = df[select]
X = df_select.drop(columns=['y'])
X.sample(10)

,age,campaign,euribor3m,cons.conf.idx,cons.price.idx
8924,37,8,4.866,-41.8,94.465
26215,47,2,4.076,-42.0,93.200
26620,51,1,4.076,-42.0,93.200
7217,38,3,4.860,-36.4,93.994
3710,52,2,4.859,-36.4,93.994
15877,40,1,4.960,-42.7,93.918
1277,24,2,4.855,-36.4,93.994
3528,36,2,4.859,-36.4,93.994
8377,46,3,4.864,-41.8,94.465
3779,35,8,4.859,-36.4,93.994


In [12]:
y = df_select.loc[:, 'y']

In [13]:
# Convert to binary classification (for XGBoost)
y = (y == 'yes').astype(int)  # Convert to binary labels

In [14]:
from sklearn.model_selection import train_test_split

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
# Save the training data as CSV
train_data = np.column_stack((y_train, X_train))  # Combine labels and features
train_df = pd.DataFrame(train_data)

test_data = np.column_stack((y_test, X_test))  # Combine labels and features
test_df = pd.DataFrame(test_data)

In [19]:
print(train_data)

[[  0.     40.      2.      4.96  -42.7    93.918]
 [  0.     31.      4.      1.244 -46.2    92.893]
 [  0.     59.      6.      1.354 -46.2    92.893]
 ...
 [  1.     35.      1.      0.754 -26.9    92.431]
 [  0.     40.      2.      4.856 -36.4    93.994]
 [  0.     29.      2.      4.96  -42.7    93.918]]


In [20]:
s3_bucket_prefix = "sagemaker/DEMO-uci_bank_marketing"
default_bucket_prefix = sagemaker_session.default_bucket_prefix

# Build prefix path
if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{s3_bucket_prefix}"
else:
    prefix = s3_bucket_prefix

In [21]:
# Save CSV locally
train_csv = "train.csv"
train_df.to_csv(train_csv, index=False, header=False)

# Upload training data to S3
train_s3_path = sagemaker_session.upload_data(train_csv, bucket=bucket, key_prefix=prefix)
print(f"Training data uploaded to: {train_s3_path}")

Training data uploaded to: s3://sagemaker-us-east-1-664752154920/sagemaker/DEMO-uci_bank_marketing/train.csv


In [22]:
# Save CSV locally
test_csv = "test.csv"
test_df.to_csv(test_csv, index=False, header=False)

# Upload testing data to S3
test_s3_path = sagemaker_session.upload_data(test_csv, bucket=bucket, key_prefix=prefix)
print(f"Training data uploaded to: {test_s3_path}")

Training data uploaded to: s3://sagemaker-us-east-1-664752154920/sagemaker/DEMO-uci_bank_marketing/test.csv


In [23]:
from sagemaker.sklearn.estimator import SKLearn

train_instance = "ml.m5.large"

sk_estimator = SKLearn(
    entry_point="train_sklearn_logist.py",
    role=role,
    instance_count=1,
    instance_type=train_instance,
    framework_version="1.2-1",
    hyperparameters={
    "max-iter": 50,
    "class-weight": "balanced"
    }
)

sk_estimator.fit({"train": train_s3_path}, job_name="logistic-regression-train-job-001")

INFO:sagemaker:Creating training-job with name: logistic-regression-train-job-001


2026-04-23 09:24:42 Starting - Starting the training job...
2026-04-23 09:24:57 Starting - Preparing the instances for training...
2026-04-23 09:25:44 Downloading - Downloading the training image......../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-04-23 09:26:54,580 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-04-23 09:26:54,585 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-04-23 09:26:54,588 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-04-23 09:26:54,605 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-04-23 09:26:54,865 s

In [24]:
print(sk_estimator.model_data)

s3://sagemaker-us-east-1-664752154920/logistic-regression-train-job-001/output/model.tar.gz


In [25]:
s3bucket = sk_estimator.model_data.split("/")[2]
key = "/".join(sk_estimator.model_data.split("/")[3:])

s3.download_file(bucket, key, "model.tar.gz")

In [26]:
import tarfile

with tarfile.open("model.tar.gz") as tar:
    tar.extractall("model_dir")

/tmp/ipykernel_25953/2937514973.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("model_dir")


In [29]:
# for debugging without deployment
from train_sklearn_logist import model_fn, predict_fn

model = model_fn("model_dir/")

preds = predict_fn(X_test, model)
print(preds)

model function got called
prediction get called
[1. 0. 1. ... 0. 0. 0.]


/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [30]:
print("Weights:", model.coef_)
print("Bias:", model.intercept_)

Weights: [[ 1.93862641e-05 -5.79077720e-02 -5.32682963e-01  5.12318597e-02
   4.06024953e-02]]
Bias: [0.00076995]


In [31]:
from sagemaker.serverless import ServerlessInferenceConfig

# 1. Define the serverless configuration
# Memory choices: 1024, 2048, 3072, 4092, 5120, or 6144 MB
serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048, 
    max_concurrency=5
)

endpoint_name = "linear-classifier-v1-serverless"

# 2. Deploy using the configuration instead of instance_type
predictor = sk_estimator.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name=endpoint_name
)

print("Serverless Model deployed!")

INFO:sagemaker:Creating model with name: sagemaker-scikit-learn-2026-04-23-09-27-39-031
INFO:sagemaker:Creating endpoint-config with name linear-classifier-v1-serverless
INFO:sagemaker:Creating endpoint with name linear-classifier-v1-serverless


-------!Serverless Model deployed!


In [33]:
from sklearn.metrics import confusion_matrix, classification_report
from sagemaker.serializers import CSVSerializer


batch_size = 100
predictions = []
# expect csv, not pandas
predictor.serializer = CSVSerializer()

for i in range(0, len(X_test.values), batch_size):
    batch = X_test.values[i:i+batch_size]
    payload = "\n".join(
        [",".join(map(str, row)) for row in batch]
    )
    response = predictor.predict(
        payload,
        initial_args={"ContentType": "text/csv"}
    )

    if isinstance(response, np.ndarray):
      # If your model returns scores, they are usually in the 2nd columns
      if response.ndim > 1:
        preds = response[:, 0].tolist()
        probs = response[:, 1].tolist()
      else:
        # If the array is 1D, it's just the predictions  
        preds = response.tolist()  
        probs = [] # Or handle as needed
    else:
    # Fallback for JSON-like dictionary
      probs = [p["score"] for p in response["predictions"]]
      preds = [p["predicted_label"] for p in response["predictions"]]

    predictions.extend(preds)


In [34]:
# Compute Confusion Matrix
conf_matrix = confusion_matrix(y_test, predictions)
print("Confusion Matrix:\n", conf_matrix)

# Compute Precision, Recall, and F1-score
report = classification_report(y_test, predictions, target_names=["No (0)", "Yes (1)"])
print("Classification Report:\n", report)

Confusion Matrix:
 [[5250 2053]
 [ 277  658]]
Classification Report:
               precision    recall  f1-score   support

      No (0)       0.95      0.72      0.82      7303
     Yes (1)       0.24      0.70      0.36       935

    accuracy                           0.72      8238
   macro avg       0.60      0.71      0.59      8238
weighted avg       0.87      0.72      0.77      8238



In [36]:
print(X_test.values[0,:])

[ 57.      1.      1.299 -46.2    92.893]
